# Preprocessing dev/debug

Runs `src/data/preprocessing.py` on a handful of patients to sanity-check normalization, slice extraction and mask alignment before processing the full dataset. Requires `data/raw/` to be populated (`python src/data/download.py`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data.preprocessing import run

DEV_OUT_DIR = PROJECT_ROOT / "data" / "processed_dev"  # separate from the real data/processed/ output

In [ ]:
index_df = run(
    raw_dir=PROJECT_ROOT / "data" / "raw",
    out_dir=DEV_OUT_DIR,
    modalities=["t1", "t2", "flair"],
    image_size=256,
    patient_limit=3,
)
index_df.head()

## Sanity checks

- Every image slice should be `(3, 256, 256)` (one channel per modality).
- Mask should be binary `{0, 1}` and aligned with the image (lesion pixels should sit inside brain tissue, not background).

In [ ]:
row = index_df[index_df["has_lesion"]].iloc[0]
image = np.load(PROJECT_ROOT / row["image_path"])
mask = np.load(PROJECT_ROOT / row["mask_path"])

print("image shape:", image.shape, "dtype:", image.dtype)
print("mask shape:", mask.shape, "unique values:", np.unique(mask))

from src.utils.viz import overlay_mask
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
overlay_mask(ax, image[2], mask)  # channel 2 = flair
ax.set_title(f"{row['patient_id']} slice {row['slice_idx']}")

In [ ]:
print("Slices per patient:")
print(index_df.groupby("patient_id").agg(n_slices=("slice_idx", "count"), n_with_lesion=("has_lesion", "sum")))

Once this looks correct, run the full preprocessing over all 60 patients from the command line:

```
python src/data/preprocessing.py
```

which writes to `data/processed/` (used by `src/train.py`).